# 01. Planillas diarias de cuotaparte (CNV / CAFCI)

Baja las planillas diarias que CAFCI presenta en la CNV y las apila en una base clase × día.

- **Fuente:** https://www.cnv.gov.ar/SitioWeb/FondosComunesInversion/CuotaPartes (desde ene-2020).
- **Unidad:** una fila por clase de fondo y por día. Todos los tipos de fondo; el filtro a money market se hace al final.
- **Salidas:**
  - `Data/raw/FondosCNV/planillas/AAAAMMDD_<id>.xlsx`, crudas e intactas;
  - `Data/raw/FondosCNV/manifest_planillas.csv`, con un registro por intento, hash incluido;
  - `Data/processed/FondosCNV/planillas_todas.parquet` y `planillas_mm.parquet`.

Se puede cortar y volver a correr: lo que ya está bajado no se vuelve a pedir. El servicio de archivos de la CNV da 503 a rachas. Si pasa, se espera y se corre de nuevo la celda de descarga.

Definiciones de cada variable: `claude/diccionario_fci.md` en el proyecto.

In [ ]:
import sys, importlib
from pathlib import Path
import numpy as np
import pandas as pd

# el módulo vive en esta misma carpeta
sys.path.insert(0, str(Path.cwd()))
import fondoscnv as fc
importlib.reload(fc)

R = fc.rutas()          # raíz del repo y carpetas de datos
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
{k: str(v) for k, v in R.items()}

## 1. Índice de presentaciones

Hay casi siempre dos presentaciones por día: la de la noche y una corregida al día siguiente. La fecha de los datos sale de la descripción ('... al 23 sep. 2026'), no de la fecha del documento.

In [ ]:
s = fc.sesion()
idx = fc.indice_planillas(s)
idx.to_csv(R["raw"] / "indice_planillas.csv", index=False)

print(f"presentaciones: {len(idx):,}")
print(f"fechas de datos distintas: {idx['fecha_info'].nunique():,}")
print(f"rango: {idx['fecha_info'].min():%Y-%m-%d} a {idx['fecha_info'].max():%Y-%m-%d}")
print("sin fecha de datos:", idx["fecha_info"].isna().sum())
idx.head()

In [ ]:
sel = fc.ultima_version(idx)
print("versiones por fecha:")
print(sel["n_versiones"].value_counts().sort_index())

# días hábiles por año, para ver que no falten meses
sel.groupby(sel["fecha_info"].dt.year).size().rename("dias").to_frame().T

## 2. Descarga

Primero un piloto de unas pocas fechas repartidas en el período, para chequear que el formato no cambió. Después, todo.

In [ ]:
MODO = "piloto"   # "piloto" o "completo"

if MODO == "piloto":
    fechas_piloto = ["2020-01-03", "2021-06-15", "2021-12-30", "2022-01-03", "2023-08-25",
                     "2024-04-15", "2025-07-31", "2025-08-01", "2025-12-01", "2026-09-23"]
    objetivo = sel[sel["fecha_info"].isin(pd.to_datetime(fechas_piloto))]
else:
    objetivo = sel

ya = {p.name.split("_")[0] for p in R["raw_planillas"].glob("*.xlsx")}
faltan = objetivo[~objetivo["fecha_info"].dt.strftime("%Y%m%d").isin(ya)]
print(f"a bajar: {len(faltan):,} de {len(objetivo):,}")

In [ ]:
# ~3 pedidos por archivo; con pausa de 0,5 s son unos 25-40 minutos para todo el período
log = fc.descargar_planillas(faltan, R["raw_planillas"], R["raw"] / "manifest_planillas.csv",
                             pausa=0.5, s=s)
if len(log):
    print(log["estado"].value_counts())
    display(log[log["estado"] != "ok"].head(20))

In [ ]:
# estado general según el manifest (puede tener varios intentos por fecha)
man = pd.read_csv(R["raw"] / "manifest_planillas.csv")
ult = man.sort_values("descargado").groupby("fecha_info").tail(1)
print(ult["estado"].value_counts())
print("archivos en disco:", len(list(R["raw_planillas"].glob("*.xlsx"))))

## 3. Lectura y chequeos de formato

La planilla tiene dos filas de encabezado. En la segunda aparecen fechas que cambian cada día (día anterior, fin de mes, fin de año, hace 12 meses), así que las columnas se leen por grupo y posición, no por nombre. Las planillas de 2020–2023 tienen 44 columnas y las recientes 47: las tres nuevas son la marca del blanqueo (Ley 27.743), el tipo de money market (Clásico/Dinámico) y 'Calificado'. Donde no existen quedan vacías.

In [ ]:
archivos = sorted(R["raw_planillas"].glob("*.xlsx"))
prueba = fc.leer_planilla(archivos[0])
print(archivos[0].name, prueba.shape)
print(prueba["seccion"].value_counts().head(8))
prueba[prueba["seccion"].isin(fc.SECCIONES_MM)].head()

In [ ]:
# columnas presentes en cada archivo del piloto, para ver cambios de formato
cols = {p.name[:8]: set(fc.leer_planilla(p).columns) for p in archivos[:12]}
todas = sorted(set().union(*cols.values()))
pd.DataFrame({k: [c in v for c in todas] for k, v in cols.items()}, index=todas).astype(int)

## 4. Base apilada

Dos archivos: todas las secciones, y sólo money market en pesos, que es la base de trabajo. Con todo el período la base completa ronda los 6 millones de filas.

In [ ]:
todas = fc.apilar_planillas(R["raw_planillas"], R["proc"] / "planillas_todas.parquet")
mm = todas[todas["seccion"].isin(fc.SECCIONES_MM)].copy()
mm.to_parquet(R["proc"] / "planillas_mm.parquet", index=False)
print(f"todas: {len(todas):,} filas | money market ARS: {len(mm):,} filas")
print("errores de lectura:", (R["proc"] / "planillas_todas.errores.csv").exists())

In [ ]:
# chequeos: cantidad de fondos y clases money market por día y patrimonio total
chk = (mm[mm["desactualizado"] == 0]
       .groupby("fecha_planilla")
       .agg(fondos=("cod_cnv", "nunique"), clases=("cod_cafci", "nunique"),
            pn_billones=("pn", lambda x: x.sum() / 1e12)))
chk.tail(10)

In [ ]:
# identidad contable: PN = cuotapartes x VCP / 1000 (debería dar ~0)
x = mm[(mm["pn"] > 0)]
dif = (x["ccp"] * x["vcp"] / 1000 / x["pn"] - 1).abs()
print("clases con diferencia > 0,1%:", int((dif > 0.001).sum()), "de", len(x))